# EDA — E-Commerce Fraud Transactions (`Fraud_Data.csv`)

**Project:** Improved Detection of Fraud Cases — Adey Innovations Inc.  
**Task:** Task 1 — Data Analysis and Preprocessing  
**Dataset:** `Fraud_Data.csv` + `IpAddress_to_Country.csv`

---

## Objectives

1. Understand the structure and quality of the e-commerce transaction dataset.
2. Identify and handle missing values, duplicates, and incorrect data types.
3. Explore univariate distributions and bivariate relationships with the fraud label.
4. Quantify and visualize the class imbalance.
5. Enrich transactions with geolocation data via IP-to-country range lookup.

---

## Notebook Structure

| Section | Description |
|---------|-------------|
| 1 | Imports & Configuration |
| 2 | Load Data |
| 3 | Data Cleaning |
| 4 | Exploratory Data Analysis |
| 5 | Class Imbalance Analysis |
| 6 | Geolocation Integration |
| 7 | Save Cleaned Data |

## 1. Imports & Configuration

Set up all required libraries, configure logging, plot styling, and define 
directory paths used throughout this notebook.

In [2]:
import sys
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Logging configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s — %(levelname)s — %(message)s'
)
logger = logging.getLogger(__name__)

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Paths
RAW_DIR = '../data/raw/'
PROCESSED_DIR = '../data/processed/'

logger.info('Libraries loaded successfully.')

2026-06-07 19:02:19,016 — INFO — Libraries loaded successfully.


## 2. Load Data

Load `Fraud_Data.csv` and `IpAddress_to_Country.csv` from the raw data directory.
File loading is wrapped in error handling to catch missing files early and surface
a clear message before any downstream steps run.

In [5]:
try:
    fraud_df = pd.read_csv(RAW_DIR + 'Fraud_Data.csv')
    ip_df = pd.read_csv(RAW_DIR + 'IpAddress_to_Country.csv')
    logger.info(f'Fraud_Data loaded: {fraud_df.shape}')
    logger.info(f'IpAddress_to_Country loaded: {ip_df.shape}')
except FileNotFoundError as e:
    logger.error(f'Dataset not found: {e}')
    logger.error('Place raw CSVs in data/raw/ before running this notebook.')
    raise
except Exception as e:
    logger.error(f'Unexpected error loading data: {e}')
    raise

print('--- Fraud_Data: first 5 rows ---')
display(fraud_df.head())
print('\n--- IpAddress_to_Country: first 5 rows ---')
display(ip_df.head())

2026-06-07 19:03:12,393 — INFO — Fraud_Data loaded: (151112, 11)
2026-06-07 19:03:12,395 — INFO — IpAddress_to_Country loaded: (138846, 3)


--- Fraud_Data: first 5 rows ---


,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0



--- IpAddress_to_Country: first 5 rows ---


,lower_bound_ip_address,upper_bound_ip_address,country
0,16777216.0,16777471,Australia
1,16777472.0,16777727,China
2,16777728.0,16778239,China
3,16778240.0,16779263,Australia
4,16779264.0,16781311,China


## 3. Data Cleaning

Steps performed in this section:

- **3.1 Basic Inspection** — review shape, dtypes, and memory usage to understand
  the dataset before making any changes.
- **3.2 Missing Values** — detect columns with nulls, report count and percentage,
  and drop rows where the target variable (`class`) is missing since those records
  cannot be used for supervised learning.
- **3.3 Duplicates** — identify and remove exact duplicate rows to prevent data
  leakage and inflated evaluation metrics.
- **3.4 Fix Data Types** — parse `signup_time` and `purchase_time` as proper
  datetime objects (they arrive as strings) and cast the target to integer.

In [6]:
# --- 3.1 Basic inspection ---
print('Shape:', fraud_df.shape)
print('\nData types:')
print(fraud_df.dtypes)
print('\nMemory usage:')
print(fraud_df.memory_usage(deep=True))

Shape: (151112, 11)

Data types:
user_id             int64
signup_time           str
purchase_time         str
purchase_value      int64
device_id             str
source                str
browser               str
sex                   str
age                 int64
ip_address        float64
class               int64
dtype: object

Memory usage:
Index                  132
user_id            1208896
signup_time       10275616
purchase_time     10275616
purchase_value     1208896
device_id          9368944
source             7949672
browser            8185186
sex                7555600
age                1208896
ip_address         1208896
class              1208896
dtype: int64


In [7]:
# --- 3.2 Missing values ---
missing = fraud_df.isnull().sum()
missing_pct = (missing / len(fraud_df) * 100).round(2)
missing_report = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
missing_report = missing_report[missing_report['missing_count'] > 0]

if missing_report.empty:
    logger.info('No missing values detected in Fraud_Data.')
else:
    logger.warning(f'Missing values found:\n{missing_report}')
    if fraud_df['class'].isnull().any():
        fraud_df.dropna(subset=['class'], inplace=True)
        logger.info('Dropped rows with missing target (class).')

display(missing_report)

2026-06-07 19:04:35,056 — INFO — No missing values detected in Fraud_Data.


,missing_count,missing_%


In [ ]:
# --- 3.3 Duplicates ---
n_duplicates = fraud_df.duplicated().sum()
logger.info(f'Duplicate rows found: {n_duplicates}')

if n_duplicates > 0:
    fraud_df.drop_duplicates(inplace=True)
    logger.info(f'Duplicates removed. New shape: {fraud_df.shape}')

2026-06-07 19:04:44,919 — INFO — Duplicate rows found: 0


In [ ]:
# --- 3.4 Fix data types ---
try:
    fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
    fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])
    logger.info('Datetime columns parsed successfully.')
except Exception as e:
    logger.error(f'Failed to parse datetime columns: {e}')
    raise

fraud_df['class'] = fraud_df['class'].astype(int)

print('Updated dtypes:')
print(fraud_df.dtypes)

2026-06-07 19:04:57,840 — INFO — Datetime columns parsed successfully.


Updated dtypes:
user_id                    int64
signup_time       datetime64[us]
purchase_time     datetime64[us]
purchase_value             int64
device_id                    str
source                       str
browser                      str
sex                          str
age                        int64
ip_address               float64
class                      int64
dtype: object
